# Chapter 1: Tokenizer from characters to BPE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch01_tokenizer.ipynb)

This notebook is generated from the complete chapter source. Nothing is replaced by a toy implementation. Python files are only split into notebook cells for readability; concatenating those cells reproduces the original source exactly.

**Pinned upstream commit:** `c9b6e2ed531b08dd9f451a091a34e9645148e2e2`


## Notebook architecture

The notebook follows the chapter as a readable pipeline rather than hiding implementation behind `%run` calls. Shared local modules used by the chapter are written from visible cells first, then every chapter script is presented in source order. Original model dimensions, algorithms, and training hyperparameters are preserved.


In [ ]:
from pathlib import Path
import os
import subprocess

UPSTREAM_COMMIT = 'c9b6e2ed531b08dd9f451a091a34e9645148e2e2'
WORKDIR = Path('/content/deep-learning-from-scratch-6')

if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/oreilly-japan/deep-learning-from-scratch-6.git', str(WORKDIR)], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'checkout', '--quiet', UPSTREAM_COMMIT], check=True)

os.chdir(WORKDIR)
print('working directory:', Path.cwd())
try:
    import torch
    print('torch:', torch.__version__)
    print('cuda:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('torch check:', exc)


## Shared modules used by this chapter

These cells keep shared architecture visible while preserving the original package layout for imports.


### `codebot/tokenizer.py`

SHA-256: `dde498e0613e73005691245fe9a253b848dda5bfac53216a4da4c643697ea4aa`


In [ ]:
%%writefile codebot/tokenizer.py
import regex as re
from collections import defaultdict
import pickle
from tqdm import tqdm


def pretokenize(text):
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    return re.findall(pattern, text)

def count_pairs(ids, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids

def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):
    texts = input_text.split(end_token)

    ids_list = []
    for text in texts:
        for pretoken in pretokenize(text):
            ids_list.append(list(pretoken.encode("utf-8")))

    num_merges = vocab_size - 256 - 1
    merge_rules = {}

    for step in tqdm(range(num_merges), desc="Training BPE"):
        counts = defaultdict(int)
        for ids in ids_list:
            counts = count_pairs(ids, counts)

        if not counts:
            break

        # best_pair = max(counts, key=counts.get)
        best_pair = max(counts, key=lambda pair: (counts[pair], pair[0], pair[1]))

        new_id = 256 + step
        merge_rules[best_pair] = new_id

        for i in range(len(ids_list)):
            ids_list[i] = merge(ids_list[i], best_pair, new_id)

    return merge_rules


class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    @staticmethod
    def load_from(filepath):
        with open(filepath, "rb") as f:
            merge_rules = pickle.load(f)
        return BPETokenizer(merge_rules)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))
        for merge_pair, new_id in self.merge_rules.items():
            ids = merge(ids, merge_pair, new_id)
        return ids

    def encode(self, input_text, show_progress=False):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        # show_progressがTrueならtqdmで進捗表示
        texts = tqdm(texts, desc="Encoding") if show_progress else texts

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                # 各事前トークンをBPEエンコード
                for pretoken in pretokenize(text):
                    ids = self._encode_text(pretoken)
                    all_ids.extend(ids)

        return all_ids

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text


## Complete chapter source


## `ch01/01_char_tokenizer.py`

SHA-256: `ff96d3d45a1db9a89c6239fa2d39e81fef5e6e2909f104dd91aa532bddd28046`


**Execution**


In [ ]:
text = "hello世界😁"
print(list(text))  # ['h', 'e', 'l', 'l', 'o', '世', '界', '😁']

print(ord('h'))  # 104
print(ord('😁'))  # 128513

print(chr(104))    # 'h'
print(chr(128513)) # '😁'

ids = [ord(char) for char in list(text)]
print(ids)  # [104, 101, 108, 108, 111, 19990, 30028, 128513]



**Definitions**


In [ ]:
class CharTokenizer:
    def encode(self, text):
        return [ord(char) for char in text]

    def decode(self, ids):
        return ''.join([chr(i) for i in ids])



**Execution**


In [ ]:
tokenizer = CharTokenizer()
text = "hello世界😁"

# エンコード
ids = tokenizer.encode(text)
print(ids)  # [104, 101, 108, 108, 111, 19990, 30028, 128513]

# デコード
decoded = tokenizer.decode(ids)
print(decoded)  # hello世界😁


## `ch01/02_byte_tokenizer.py`

SHA-256: `4cfda13d9a1d185522d33ee8cae50ac6420390f67752bc46d44e1c499616ece4`


**Execution**


In [ ]:
# 'A' の場合
encoded = 'A'.encode("utf-8")
print(encoded)        # b'A'
print(list(encoded))  # [65]

# 'あ' の場合
encoded = 'あ'.encode("utf-8")
print(encoded)        # b'\xe3\x81\x82'
print(list(encoded))  # [227, 129, 130]

ids = [65]
decoded = bytes(ids).decode("utf-8")
print(decoded)   # 'A'




**Definitions**


In [ ]:
class ByteTokenizer:
    def encode(self, text):
        return list(text.encode("utf-8"))

    def decode(self, ids):
        return bytes(ids).decode("utf-8")


# 使用例


**Execution**


In [ ]:
tokenizer = ByteTokenizer()
text = "hello世界😁"

# エンコード
ids = tokenizer.encode(text)
print(ids)  # [104, 101, 108, 108, 111, 228, 184, 150, 231, 149, 140, 240, 159, 152, 129]

# デコード
decoded = tokenizer.decode(ids)
print(decoded)  # hello世界😁


## `ch01/03_bpe_train.py`

SHA-256: `bbe9cd6145887f93acd12189cb2ac1cf826e8e3e79395d2e725652edbc14c8a8`


**Imports**


In [ ]:
from collections import defaultdict



**Definitions**


In [ ]:
def count_pairs(ids):
    counts = defaultdict(int)
    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts

# 使用例


**Execution**


In [ ]:
ids = [1, 2, 3, 1, 2]
counts = count_pairs(ids)
print(counts)  # {(1, 2): 2, (2, 3): 1, (3, 1): 1}



**Definitions**


In [ ]:
def merge(ids, pair, new_id):
    merged_ids = []
    i = 0

    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1

    return merged_ids

# 使用例


**Execution**


In [ ]:
ids = [1, 2, 3, 1, 2]
merged = merge(ids, (1, 2), 4)
print(merged)  # [4, 3, 4]



**Definitions**


In [ ]:
def train_bpe(text, vocab_size):
    # テキストを0~255のID列に変換
    ids = list(text.encode("utf-8"))

    # マージ回数を決定
    num_merges = vocab_size - 256  # 256は初期の語彙サイズ
    merge_rules = {}

    for step in range(num_merges):
        # 隣接ペアの統計を取得
        counts = count_pairs(ids)

        # ペアが存在しない場合の処理
        if not counts:
            break

        # 最頻出ペアを選択
        best_pair = max(counts, key=counts.get)
        # best_pair = max(counts, key=lambda pair: (counts[pair], pair[0], pair[1]))

        # 新しいトークンIDを割り当て
        new_id = 256 + step
        merge_rules[best_pair] = new_id

        # マージを実行
        ids = merge(ids, best_pair, new_id)

    return merge_rules

# 使用例


**Execution**


In [ ]:
text = "Hello world! This is BPE training."
merge_rules = train_bpe(text, vocab_size=260)
print(merge_rules)  # {(105, 115): 256, (256, 32): 257, (105, 110): 258, (72, 101): 259}


## `ch01/04_bpe_tokenizer.py`

SHA-256: `d3d5b4b2c426fd7207f3bc1280902f3d4b019e74c1bb9c3c0726db8b7e87b06a`


**Imports**


In [ ]:
from collections import defaultdict



**Definitions**


In [ ]:
def count_pairs(ids):
    counts = defaultdict(int)
    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0

    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1

    return merged_ids


class BPETokenizer:
    def __init__(self, merge_rules):
        self.merge_rules = merge_rules

        # IDからバイト列への対応表（0~255を登録）
        self.id_to_bytes = {i: bytes([i]) for i in range(256)}

        # マージされたトークンは元のトークンのバイト列を連結
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]

        # 語彙サイズを設定
        self.vocab_size = len(self.id_to_bytes)

    def encode(self, text):
        ids = list(text.encode("utf-8"))

        # 学習時の順序でマージルールを適用
        for merge_pair, new_id in self.merge_rules.items():
            ids = merge(ids, merge_pair, new_id)

        return ids

    def decode(self, ids):
        # 各トークンIDを対応するバイト列に変換
        byte_list = [self.id_to_bytes[i] for i in ids]

        # すべてのバイト列を連結
        combined_bytes = b"".join(byte_list)

        # バイト列をUTF-8テキストに変換
        text = combined_bytes.decode("utf-8", errors="replace")
        return text

# 学習済みのマージルール


**Execution**


In [ ]:
merge_rules = {(105, 115): 256, (256, 32): 257, (105, 110): 258, (72, 101): 259}

# トークナイザーを作成
tokenizer = BPETokenizer(merge_rules)

# テキストをエンコード
text = "Hello世界😁"
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print(ids)  # [259, 108, 108, 111, 228, 184, 150, 231, 149, 140, 240, 159, 152, 129]
print(decoded)  # Hello世界😁


## `ch01/05_special_token.py`

SHA-256: `a0b782fd01baf307a58fd6fe72bbfcb8a7f6920db506dcd7938c1a1b65389db2`


**Imports**


In [ ]:
from collections import defaultdict
import re

# def count_pairs(ids):
#     counts = defaultdict(int)
#     for pair in zip(ids, ids[1:]):
#         counts[pair] += 1
#     return counts



**Definitions**


In [ ]:
def count_pairs(ids, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids

def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):
    # 特殊トークンでテキストを分割
    texts = input_text.split(end_token)
    ids_list = [list(text.encode("utf-8")) for text in texts]

    # 基本語彙（0-255）+ 終了トークン用（1個）を除いた分がマージ回数
    num_merges = vocab_size - 256 - 1
    merge_rules = {}

    for step in range(num_merges):
        # 隣接ペアの頻度を集計
        counts = defaultdict(int)
        for ids in ids_list:
            counts = count_pairs(ids, counts)

        # ペアが存在しない場合の処理
        if not counts:
            break

        # 最頻出ペアを選択
        best_pair = max(counts, key=counts.get)
        # best_pair = max(counts, key=lambda pair: (counts[pair], pair[0], pair[1]))
        new_id = 256 + step
        merge_rules[best_pair] = new_id

        # マージを実行
        for i in range(len(ids_list)):
            ids_list[i] = merge(ids_list[i], best_pair, new_id)

    return merge_rules

# 使用例


**Execution**


In [ ]:
sample_text = "Hello world!<|endoftext|>This is BPE training."

merge_rules = train_bpe(sample_text, vocab_size=260)
print(merge_rules)  # {(105, 115): 256, (256, 32): 257, (105, 110): 258}




**Definitions**


In [ ]:
class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))
        for merge_pair, new_id in self.merge_rules.items():
            ids = merge(ids, merge_pair, new_id)
        return ids

    def encode(self, input_text):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                ids = self._encode_text(text)
                all_ids.extend(ids)

        return all_ids

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text




**Execution**


In [ ]:
tokenizer = BPETokenizer(merge_rules)

text = "Hello world!<|endoftext|>"
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print(ids)
print(decoded)


## `ch01/06_pretokenize.py`

SHA-256: `618aaff5b0ae7e5d77971cfe4cda0528f7bef78b3409a0d78b8aedafc9863746`


**Imports**


In [ ]:
from collections import defaultdict
import regex as re
from tqdm import tqdm




**Definitions**


In [ ]:
def pretokenize(text):
    # GPT-2で使用されている正規表現パターン
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    return re.findall(pattern, text)

def count_pairs(ids, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += 1
    return counts

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids

def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):
    # ステップ1: 特殊トークンで分割
    texts = input_text.split(end_token)

    # ステップ2: 各テキスト片を事前トークン化
    ids_list = []
    for text in texts:
        for pretoken in pretokenize(text):  # 事前トークン化
            ids_list.append(list(pretoken.encode("utf-8")))  # ID列に変換

    # ==== 残りは元のコードと同じ（ただしtqdmを追加） ====
    num_merges = vocab_size - 256 - 1
    merge_rules = {}

    for step in tqdm(range(num_merges), desc="Training BPE"):  # tqdmで進捗表示
        counts = defaultdict(int)
        for ids in ids_list:
            counts = count_pairs(ids, counts)

        if not counts:
            break

        best_pair = max(counts, key=counts.get)
        # best_pair = max(counts, key=lambda pair: (counts[pair], pair[0], pair[1]))

        new_id = 256 + step
        merge_rules[best_pair] = new_id

        for i in range(len(ids_list)):
            ids_list[i] = merge(ids_list[i], best_pair, new_id)

    return merge_rules


class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))
        for merge_pair, new_id in self.merge_rules.items():
            ids = merge(ids, merge_pair, new_id)
        return ids

    def encode(self, input_text, show_progress=False):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        # show_progressがTrueならtqdmで進捗表示
        texts = tqdm(texts, desc="Encoding") if show_progress else texts

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                # 各事前トークンをBPEエンコード
                for pretoken in pretokenize(text):
                    ids = self._encode_text(pretoken)
                    all_ids.extend(ids)

        return all_ids

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text

# 事前トークン化対応のBPE学習


**Execution**


In [ ]:
sample_text = "Say hello! Why hello? Just hello.<|endoftext|>Good morning!"

merge_rules = train_bpe(sample_text, vocab_size=270)
tokenizer = BPETokenizer(merge_rules)

# エンコード/デコード
text = "Say hello!"
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print(ids)
print(decoded)

# 各トークンIDをデコードして確認
for token_id in ids:
    print(f"{token_id} -> '{tokenizer.decode([token_id])}'")


## `ch01/07_tiny_codes.py`

SHA-256: `e1a28dd0ff82e84ff1100556c2961a1a345ea4f5ccb7311ccbfb8bf2d085e65c`


**Imports**


In [ ]:
import os, sys


**Execution**


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
sys.path.append('.')



**Imports**


In [ ]:
import pickle
from codebot.tokenizer import train_bpe



**Execution**


In [ ]:
vocab_size = 1000  # 語彙サイズ
text = open("codebot/tiny_codes.txt").read()
merge_rules = train_bpe(text, vocab_size)

# 学習済みマージルールをファイルに保存
with open("codebot/merge_rules.pkl", "wb") as f:
    pickle.dump(merge_rules, f)


## `ch01/08_eval.py`

SHA-256: `7c9b1f4c293bd7dd273a9cac145941a9f0abc6f1044d1fe0e88a988baf1c6246`


**Imports**


In [ ]:
import os, sys


**Execution**


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
sys.path.append('.')



**Imports**


In [ ]:
from codebot.tokenizer import BPETokenizer




**Execution**


In [ ]:
tokenizer = BPETokenizer.load_from("codebot/merge_rules.pkl")

print("最初に学習された10個:")
for token_id in range(256, 266):
    byte_seq = tokenizer.id_to_bytes[token_id]
    text = byte_seq.decode("utf-8")
    print(f"  ID {token_id}: '{text}'")

print("\n最後に学習された10個:")
for token_id in range(990, 1000):
    byte_seq = tokenizer.id_to_bytes[token_id]
    text = byte_seq.decode("utf-8")
    print(f"  ID {token_id}: '{text}'")


# 圧縮率を測定
sample_text = open("codebot/tiny_codes.txt").read()[:10000]  # 最初の10000文字

byte_count = len(sample_text.encode("utf-8"))
ids = tokenizer.encode(sample_text)
ids_count = len(ids)
compression_ratio = byte_count / ids_count

print("\n=== 圧縮効率 ===")
print(f"バイト数: {byte_count:,}")
print(f"トークン数: {ids_count:,}")
print(f"圧縮率: {compression_ratio:.2f}倍（平均 {compression_ratio:.2f} バイト/トークン）")


# ==== 以下はGPT系モデルのエンコードと圧縮率比較（tiktoken使用） ====
"""
import tiktoken

text = open("codebot/tiny_codes.txt").read()[:10000]
byte_count = len(text.encode("utf-8"))

for name, encoding_name in [('GPT-2', 'gpt2'), ('cl100k_base', 'cl100k_base')]:
    encoding = tiktoken.get_encoding(encoding_name)
    token_count = len(encoding.encode(text, allowed_special={'<|endoftext|>'}))
    ratio = byte_count / token_count
    print(f"{name}: 語彙サイズ {encoding.n_vocab:,}, 圧縮率 {ratio:.2f}倍")
"""


## `ch01/09_bpe_encode.py`

SHA-256: `d8e8b90a768d75453fb38afe4c7fee206e0a6a1616728f2de88e83fbdac00f81`


**Imports**


In [ ]:
import os, sys


**Execution**


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))
sys.path.append('.')



**Imports**


In [ ]:
import numpy as np
from codebot.tokenizer import BPETokenizer


# トークナイザを読み込み


**Execution**


In [ ]:
tokenizer = BPETokenizer.load_from("codebot/merge_rules.pkl")

# テキストをトークンIDに変換（進捗バーを表示）
text = open("codebot/tiny_codes.txt").read()
ids = tokenizer.encode(text, show_progress=True)

# numpy配列に変換して保存
ids_array = np.array(ids, dtype=np.uint16)
ids_array.tofile("codebot/tiny_codes.bin")

print(f"トークンID数: {len(ids_array)}")
print(f"最初の20個のトークンID: {ids_array[:20]}")


## T4 execution note

The implementation above keeps the upstream code and hyperparameters intact. For chapters with long training loops, a Colab T4 can execute the implementation, but completing the full training schedule may take substantial wall-clock time. No reduced model, shortened algorithm, or toy substitute is enabled by default in this notebook.
